Add interactive graphs and sliders for inputs by implementing custom visualization components, ensuring compatibility with multiple device types, and dynamically linking input changes to real-time model updates.

In [1]:
import pandas as pd
import numpy as np
import joblib
import json
import ipywidgets as widgets
import plotly.graph_objects as go

from IPython.display import display, clear_output

In [2]:
df = pd.read_csv("heart_disease_train.csv")

model = joblib.load("saved_models/heart_disease_model_v1.joblib")

print("Dataset loaded:", df.shape)
print("Model loaded successfully!")

Dataset loaded: (79, 19)
Model loaded successfully!


In [3]:
if hasattr(model, "feature_names_in_"):
    features = list(model.feature_names_in_)
else:
    with open("saved_models/heart_disease_model_v1_metadata.json") as f:
        metadata = json.load(f)

    features = metadata.get("features", metadata.get("feature_names"))

if not features:
    raise ValueError("Model features not found.")

classes = list(model.classes_)

print("Model Features:", features)
print("Model Classes:", classes)

Model Features: ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
Model Classes: [np.int64(0), np.int64(1)]


In [4]:
input_widgets = {}

for feature in features:

    if pd.api.types.is_numeric_dtype(df[feature]):

        minimum = float(df[feature].min())
        maximum = float(df[feature].max())
        default = float(df[feature].median())

        if minimum == maximum:
            maximum = minimum + 1

        if pd.api.types.is_integer_dtype(df[feature]):
            input_widgets[feature] = widgets.IntSlider(
                value=int(default),
                min=int(minimum),
                max=int(maximum),
                description=feature,
                continuous_update=False,
                style={"description_width": "120px"},
                layout=widgets.Layout(width="100%")
            )
        else:
            input_widgets[feature] = widgets.FloatSlider(
                value=default,
                min=minimum,
                max=maximum,
                step=0.1,
                description=feature,
                continuous_update=False,
                style={"description_width": "120px"},
                layout=widgets.Layout(width="100%")
            )

    else:
        categories = sorted(df[feature].dropna().unique().tolist())

        input_widgets[feature] = widgets.Dropdown(
            options=categories,
            value=categories[0],
            description=feature,
            style={"description_width": "120px"},
            layout=widgets.Layout(width="100%")
        )

input_layout = widgets.VBox(
    list(input_widgets.values()),
    layout=widgets.Layout(width="100%", max_width="700px")
)

display(input_layout)

In [5]:
output = widgets.Output()

def update_prediction(change=None):

    with output:
        clear_output(wait=True)

        user_data = {
            feature: input_widgets[feature].value
            for feature in features
        }

        input_df = pd.DataFrame([user_data], columns=features)

        try:
            probabilities = model.predict_proba(input_df)[0]
            prediction = model.predict(input_df)[0]

            print("Predicted Class:", prediction)
            print("Prediction Probabilities:")

            for cls, probability in zip(classes, probabilities):
                print(f"Class {cls}: {probability * 100:.2f}%")

            fig = go.Figure()

            fig.add_trace(go.Bar(
                x=[str(cls) for cls in classes],
                y=probabilities * 100,
                text=[f"{p * 100:.2f}%" for p in probabilities],
                textposition="auto"
            ))

            fig.update_layout(
                title="Real-Time Prediction Confidence",
                xaxis_title="Prediction Class",
                yaxis_title="Probability (%)",
                yaxis=dict(range=[0, 100]),
                template="plotly_white",
                autosize=True,
                height=400
            )

            display(fig)

            input_fig = go.Figure()

            input_fig.add_trace(go.Bar(
                x=features,
                y=[float(user_data[f]) for f in features]
            ))

            input_fig.update_layout(
                title="Current User Input Values",
                xaxis_title="Features",
                yaxis_title="Input Values",
                template="plotly_white",
                height=400,
                autosize=True
            )

            display(input_fig)

        except Exception as e:
            print("Prediction Error:", e)


for widget in input_widgets.values():
    widget.observe(update_prediction, names="value")

display(output)

update_prediction()

Output()

In [6]:
def reset_inputs(button):

    for feature, widget in input_widgets.items():

        if isinstance(widget, widgets.IntSlider):
            widget.value = int(df[feature].median())

        elif isinstance(widget, widgets.FloatSlider):
            widget.value = float(df[feature].median())

        elif isinstance(widget, widgets.Dropdown):
            widget.value = widget.options[0]


reset_button = widgets.Button(
    description="Reset Inputs",
    button_style="warning",
    icon="refresh"
)

reset_button.on_click(reset_inputs)

display(reset_button)

Button(button_style='warning', description='Reset Inputs', icon='refresh', style=ButtonStyle())